In [1]:
import pandas as pd
from data import build_all
from pitch_suggestions import suggest_pitches

In [2]:
import importlib
import data

importlib.reload(data)

<module 'data' from '/Users/kids/Pitcher Similarity/data.py'>

In [3]:
data = build_all(live=False)

statcast_clean  = data['statcast_clean']
pitch_type_summ = data['pitch_type_summ']
pitcher_summ    = data['pitcher_summ']
pitcher_summ_r  = data['pitcher_summ_r']
pitcher_summ_l  = data['pitcher_summ_l']
pitch_type_r  = data['pitch_type_r']
pitch_type_l  = data['pitch_type_l']

statcast_clean_25 = statcast_clean[statcast_clean['game_year'] == 2025]

In [4]:
pitcher_summ_l_2125 = pitcher_summ_l[pitcher_summ_l['game_year'] <= 2025]
pitcher_summ_r_2125 = pitcher_summ_r[pitcher_summ_r['game_year'] <= 2025]
pitch_type_summ_2125 = pitch_type_summ[pitch_type_summ['game_year'] <= 2025]

## Biomechanical consistency: 2025 vs. 2026

Identify pitchers (by the `pitcher` column) who have a row in **both** `game_year == 2025` and
`game_year == 2026`, and whose 2025 and 2026 biomechanics are close enough to qualify as a
biomechanical comp for each other (standardized-Euclidean `distance <= biomech_distance_threshold`).

Distance is computed exactly as in `pitch_suggestions._find_biomech_comps`: pairwise standardized
Euclidean distance over `BIOMECH_FEATURES`, z-scored across all qualifying pitcher-years.

In [5]:
from distances import compute_euclidean_distances
from pitch_suggestions import BIOMECH_FEATURES

biomech_distance_threshold = 1.5

# Pairwise standardized-Euclidean biomech distances across all pitcher-years.
# Carrying `pitcher` in label_cols lets us match a pitcher to their own other-year row.
biomech_dist = compute_euclidean_distances(
    pitcher_summ,
    features=BIOMECH_FEATURES,
    label_cols=['pitcher', 'player_name', 'game_year'],
    min_pitches=20,
)

In [6]:
# Keep self-pairs (same pitcher) that span the 2025 and 2026 seasons.
same_pitcher = biomech_dist['pitcher1'] == biomech_dist['pitcher2']
spans_25_26  = (
    ((biomech_dist['game_year1'] == 2025) & (biomech_dist['game_year2'] == 2026)) |
    ((biomech_dist['game_year1'] == 2026) & (biomech_dist['game_year2'] == 2025))
)

self_pairs = biomech_dist[same_pitcher & spans_25_26].copy()

# A pitcher is biomechanically consistent if their 2025 and 2026 rows are within threshold.
consistent_pitchers = (
    self_pairs[self_pairs['distance'] <= biomech_distance_threshold]
    .rename(columns={'pitcher1': 'pitcher', 'player_name1': 'player_name'})
    [['pitcher', 'player_name', 'distance']]
    .sort_values('distance')
    .reset_index(drop=True)
)

print(f"{len(self_pairs)} pitchers appeared in 2025 and 2026; \n"
      f"{len(consistent_pitchers)} biomechanically consistent (distance <= {biomech_distance_threshold}).")
consistent_pitchers

440 pitchers appeared in 2025 and 2026; 
435 biomechanically consistent (distance <= 1.5).


,pitcher,player_name,distance
0,596133,"Weaver, Luke",0.104926
1,663559,"Falter, Bailey",0.106427
2,657277,"Webb, Logan",0.122373
3,676467,"Gordon, Colton",0.139905
4,687911,"King, Bryan",0.155099
...,...,...,...
430,666808,"Doval, Camilo",1.264625
431,676974,"Meyer, Max",1.294387
432,668933,"Ashcraft, Graham",1.302719
433,608337,"Giolito, Lucas",1.368006


## Novel 2026 pitches thrown by the biomechanically consistent pitchers

For each of the 434 consistent pitchers, treat their **2025 arsenal** as the target and their
**2026 pitches** as candidates, then apply `_tag_novelty` exactly as in `pitch_suggestions.py`:
a 2026 pitch is *novel* when its minimum standardized-Euclidean distance (over `PITCH_CHAR_FEATURES`)
to any 2025 pitch is `>= novelty_distance_threshold`. The scaler is fit on all pitch-characteristic
rows, identical to `suggest_pitches`. Candidates are filtered to `n >= min_pitches` so a novel pitch
reflects a real new shape rather than a handful of misclassified pitches.

In [7]:
from pitch_suggestions import PITCH_CHAR_FEATURES, _tag_novelty
from sklearn.preprocessing import StandardScaler

novelty_distance_threshold = 1.2  
min_pitches                = 20

# Global scaler fit exactly as in suggest_pitches: on every pitch-characteristic row.
global_scaler = StandardScaler().fit(
    pitch_type_summ[PITCH_CHAR_FEATURES].dropna().values
)

novel_rows = []
for pid in consistent_pitchers['pitcher'].unique():
    target_pitches = (                                  # 2025 arsenal
        pitch_type_summ[(pitch_type_summ['pitcher'] == pid) &
                        (pitch_type_summ['game_year'] == 2025)]
        .dropna(subset=PITCH_CHAR_FEATURES).reset_index(drop=True)
    )
    cand_2026 = (                                       # candidate 2026 pitches
        pitch_type_summ[(pitch_type_summ['pitcher'] == pid) &
                        (pitch_type_summ['game_year'] == 2026) &
                        (pitch_type_summ['n'] >= min_pitches)]
        .dropna(subset=PITCH_CHAR_FEATURES).reset_index(drop=True)
    )
    if target_pitches.empty or cand_2026.empty:
        continue

    _, novel = _tag_novelty(
        target_pitches, cand_2026, PITCH_CHAR_FEATURES,
        novelty_distance_threshold, global_scaler,
    )
    novel_rows.append(novel)

novel_2026 = (
    pd.concat(novel_rows, ignore_index=True) if novel_rows else pd.DataFrame()
)

In [8]:
n_pitchers = novel_2026['pitcher'].nunique()
print(f"{len(novel_2026)} novel pitches thrown by {n_pitchers} of the "
      f"{consistent_pitchers['pitcher'].nunique()} biomechanically consistent pitchers.")

novel_2026[[
    'pitcher', 'player_name', 'pitch_type', 'release_speed', 'pfx_x', 'pfx_z',
    'n', 'min_dist_to_target', 'closest_target_pitch',
]].sort_values('min_dist_to_target', ascending=False).reset_index(drop=True)

32 novel pitches thrown by 30 of the 435 biomechanically consistent pitchers.


,pitcher,player_name,pitch_type,release_speed,pfx_x,pfx_z,n,min_dist_to_target,closest_target_pitch
0,679358,"Orze, Eric",CU,82.218750,1.274375,-0.464583,48,2.103032,SL
1,573124,"Rogers, Taylor",FC,87.857746,0.008732,0.451972,71,1.813046,SI
2,669160,"May, Dustin",CU,83.192771,0.816867,-1.337590,83,1.807221,ST
3,670059,"Holderman, Colin",CU,83.021053,1.169737,-0.827632,38,1.789100,ST
4,657612,"Hill, Tim",SL,76.111905,-0.370000,0.677143,42,1.785194,SL
5,677865,"Bruihl, Justin",CH,80.560377,0.950566,-0.056038,53,1.774862,SI
6,683232,"Mears, Nick",CH,84.491667,-1.312222,0.181111,36,1.721676,SL
7,664854,"Helsley, Ryan",FS,89.940909,-1.033182,0.854091,22,1.670592,FC
8,647336,"Soroka, Michael",FC,88.984459,0.125068,0.612500,148,1.599639,FF
9,640455,"Manaea, Sean",FC,84.145000,-0.210167,0.573333,60,1.502454,ST


## Did `suggest_pitches` predict the actual novel pitches?

Run `suggest_pitches` only for the pitchers who actually threw a novel pitch — each from their own
handedness pool restricted to 2021–2025, so this is a genuine forward test. For every novel pitch we find that pitcher's **closest suggestion centroid** in
standardized `PITCH_CHAR_FEATURES` space (the same scaler used for novelty), and record the match
distance, the suggestion strength (`n_comps`), and whether the pitch type agrees.

In [9]:
import numpy as np
from pitch_suggestions import _full_name, suggest_pitches

# Suggestions only for the pitchers who threw a novel pitch, each from their own
# 2021-2025 handedness pool.
pools  = {'L': pitcher_summ_l_2125, 'R': pitcher_summ_r_2125}
throws = pitcher_summ.groupby('pitcher')['p_throws'].first().to_dict()

sugg = pd.concat([
    suggest_pitches(
        pit_id, pools[throws[pit_id]], pitch_type_summ_2125,
        min_pitches=20, biomech_distance_threshold=1.5,
        novelty_distance_threshold=1.2, min_comp_usage_pct=0.01,
    )['suggestions'].assign(target_pitcher=pit_id)
    for pit_id in novel_2026['pitcher'].unique()
], ignore_index=True)

# Closest suggestion to each actual novel pitch, in the standardized space used for novelty.
SUGG_CENTROID = ['wavg_release_speed', 'wavg_pfx_x', 'wavg_pfx_z']
novel = novel_2026.reset_index(drop=True)
novel['novel_id'] = novel.index
pairs = novel.merge(sugg, left_on='pitcher', right_on='target_pitcher', how='left')

has = pairs['target_pitcher'].notna()
pairs['sugg_match_dist'] = np.nan
pairs.loc[has, 'sugg_match_dist'] = np.linalg.norm(
    global_scaler.transform(pairs.loc[has, PITCH_CHAR_FEATURES].values)
    - global_scaler.transform(pairs.loc[has, SUGG_CENTROID].values), axis=1)

match = (
    pairs.sort_values('sugg_match_dist', na_position='last')
    .drop_duplicates('novel_id', keep='first')
    .sort_values('novel_id')
    .reset_index(drop=True)
)

In [10]:
# Headline: how often, and how closely, did a suggestion match the actual novel pitch?
hit = match['sugg_match_dist'].notna()
pd.Series({
    
    'novel_pitches':     len(match),
    'with_suggestion':   int(hit.sum()),
    'within_0.75':       int((match['sugg_match_dist'] <= 0.75).sum()),
    'within_1.2':        int((match['sugg_match_dist'] <= 1.2).sum()),
})

novel_pitches      32
with_suggestion    28
within_0.75        18
within_1.2         23
dtype: int64

In [11]:
# Were stronger suggestions (more comps) closer to what was actually thrown?
strength = match[hit].copy()
strength['matched'] = strength['sugg_match_dist'] <= 1.2
strength.groupby(pd.qcut(strength['n_comps'], 4, duplicates='drop'), observed=True).agg(
    n=('matched', 'size'),
    mean_match_dist=('sugg_match_dist', 'mean'),
    match_rate=('matched', 'mean'),
).round(2)

,n,mean_match_dist,match_rate
n_comps,,,
"(1.999, 11.25]",7,0.80,0.71
"(11.25, 18.0]",7,0.84,0.71
"(18.0, 43.75]",7,0.64,1.00
"(43.75, 159.0]",7,0.68,0.86


In [12]:
# Side-by-side: each actual novel pitch vs. its nearest suggestion.
match_view = (
    match[[
        'player_name', 'pitch_type', 'release_speed', 'pfx_x', 'pfx_z', 'n', 'min_dist_to_target',
        'cluster_label', 'n_comps', 'wavg_release_speed', 'wavg_pfx_x', 'wavg_pfx_z',
        'sugg_match_dist',
    ]]
    .rename(columns={
        'pitch_type': 'novel_type', 'n': 'novel_n', 'min_dist_to_target': 'novelty_dist',
        'cluster_label': 'sugg_cluster',
        'wavg_release_speed': 'sugg_speed', 'wavg_pfx_x': 'sugg_pfx_x', 'wavg_pfx_z': 'sugg_pfx_z',
    })
    .sort_values('sugg_match_dist', na_position='last')
    .reset_index(drop=True)
)
match_view

,player_name,novel_type,release_speed,pfx_x,pfx_z,novel_n,novelty_dist,sugg_cluster,n_comps,sugg_speed,sugg_pfx_x,sugg_pfx_z,sugg_match_dist
0,"Soroka, Michael",FC,88.984459,0.125068,0.612500,148,1.599639,Cutter*,130.0,89.0,0.18,0.65,0.082050
1,"Leiter, Jack",FC,93.029851,0.080075,0.937836,134,1.381635,Cutter*,15.0,92.3,0.12,0.90,0.141234
2,"Aldegheri, Sam",FC,86.340000,-0.132000,0.493333,45,1.262093,Cutter*,15.0,86.1,-0.23,0.63,0.229202
3,"Rogers, Taylor",FC,87.857746,0.008732,0.451972,71,1.813046,Slider*,40.0,86.8,-0.18,0.39,0.290353
4,"Helsley, Ryan",FS,89.940909,-1.033182,0.854091,22,1.670592,Changeup*,7.0,90.3,-1.15,0.66,0.314801
5,"Pallante, Andre",FS,87.648649,-0.751081,0.577838,37,1.447551,Changeup*,9.0,88.2,-1.00,0.49,0.320418
6,"Bruihl, Justin",FF,91.634091,0.482273,1.031591,44,1.207820,Four-Seam Fastball*,20.0,90.7,0.63,1.21,0.343737
7,"Manaea, Sean",FC,84.145000,-0.210167,0.573333,60,1.502454,Cutter*,2.0,82.0,-0.18,0.53,0.366460
8,"Mejia, Juan",CH,93.035000,-1.062500,0.596500,20,1.389727,Sinker*,57.0,95.1,-1.25,0.60,0.405118
9,"Holderman, Colin",CU,83.021053,1.169737,-0.827632,38,1.789100,Curveball*,59.0,80.9,0.90,-0.88,0.473053


## Visualize one pitcher: actual novel pitch vs. suggestions

The same Batter-View break plot the app shows (comp pitches colored by velocity, suggestion centroids,
existing arsenal as black diamonds, arm-angle ray) — with the **actual 2026 novel pitch overlaid as a
red star**. Alongside it, the app's Pitcher Profile and the Arsenal & Suggestions table, with the
actual 2026 pitch appended so its velo / break sit next to the suggested shapes. Call
`validate_pitcher(name)` for any pitcher in `novel_2026`.

In [13]:
import plotly.graph_objects as go
from IPython.display import display

_MARKERS = ['circle', 'square', 'triangle-up', 'diamond', 'cross',
            'x', 'triangle-down', 'triangle-left', 'triangle-right', 'hexagon']


def make_validation_fig(result, actual, is_righty):
    """App-style Batter-View break plot with the actual 2026 novel pitch overlaid as a red star."""
    comp   = result['comp_pitches'].reset_index(drop=True)
    target = result['target_pitches']
    name   = target['player_name'].iloc[0]
    keys   = sorted(comp[['cluster_label', 'cluster']].drop_duplicates().itertuples(index=False, name=None))
    vmin, vmax = comp['release_speed'].min(), comp['release_speed'].max()

    fig = go.Figure()
    for i, (label, cid) in enumerate(keys):                                  # comp pitches, colored by velo
        g = comp[(comp['cluster_label'] == label) & (comp['cluster'] == cid)]
        fig.add_trace(go.Scatter(
            x=g['pfx_x'], y=g['pfx_z'], mode='markers', name=f'Possible ({_full_name(label)})',
            marker=dict(symbol=_MARKERS[i % len(_MARKERS)], size=8, color=g['release_speed'],
                        colorscale='plasma', cmin=vmin, cmax=vmax, opacity=0.7, showscale=(i == 0),
                        colorbar=dict(title=dict(text='Velo (mph)', side='right'), x=1.02, thickness=15, len=0.75) if i == 0 else None),
            customdata=np.column_stack([g['player_name'], g['pitch_type'].map(_full_name), g['release_speed']]),
            hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]} — %{customdata[2]:.1f} mph<extra></extra>'))

    cen = comp.groupby(['cluster_label', 'cluster'])[['pfx_x', 'pfx_z', 'release_speed']].mean().reset_index()
    for i, (label, cid) in enumerate(keys):                                  # suggestion centroids
        r = cen[(cen['cluster_label'] == label) & (cen['cluster'] == cid)].iloc[0]
        fig.add_trace(go.Scatter(
            x=[r['pfx_x']], y=[r['pfx_z']], mode='markers', name='Suggestion Centroid',
            showlegend=(i == 0), legendgroup='centroid',
            marker=dict(symbol=_MARKERS[i % len(_MARKERS)], size=16, color=[r['release_speed']],
                        colorscale='plasma', cmin=vmin, cmax=vmax, line=dict(color='black', width=2)),
            hovertemplate=f'<b>Suggestion: {_full_name(label)}</b><br>HB %{{x:.2f}} / IVB %{{y:.2f}}<extra></extra>'))

    fig.add_trace(go.Scatter(                                                # existing arsenal
        x=target['pfx_x'], y=target['pfx_z'], mode='markers+text', name='Existing Pitch',
        marker=dict(symbol='diamond', size=15, color='black'),
        text=[_full_name(p) for p in target['pitch_type']], textposition='top right', textfont=dict(size=11, color='black'),
        hovertemplate='Existing: %{text}<extra></extra>'))

    fig.add_trace(go.Scatter(                                                # the actually-thrown 2026 novel pitch
        x=actual['pfx_x'], y=actual['pfx_z'], mode='markers+text', name='Actual 2026 Novel',
        marker=dict(symbol='star', size=24, color='red', line=dict(color='black', width=1.5)),
        text=[_full_name(p) for p in actual['pitch_type']], textposition='bottom center', textfont=dict(size=12, color='red'),
        customdata=np.column_stack([actual['pitch_type'].map(_full_name), actual['release_speed'], actual['min_dist_to_target']]),
        hovertemplate='<b>ACTUAL 2026: %{customdata[0]}</b><br>%{customdata[1]:.1f} mph<br>HB %{x:.2f} / IVB %{y:.2f}<br>novelty %{customdata[2]:.2f}<extra></extra>'))

    ang = result['target_info']['arm_angle']                                # arm-angle ray (mirror RHP)
    rad, d = np.radians(ang), (-1 if is_righty else 1)
    fig.add_trace(go.Scatter(x=[0, d * 1.5 * np.cos(rad)], y=[0, 1.5 * np.sin(rad)], mode='lines',
        name='Arm Angle', line=dict(color='rgba(50,50,50,0.3)', width=6),
        hovertemplate=f'Arm Angle {ang:.1f}°<extra></extra>'))

    grid = dict(showgrid=True, gridcolor='lightgrey', zeroline=True, zerolinecolor='darkgrey', range=[-2.1, 2.1], constrain='domain')
    fig.update_layout(
        title=dict(text=f'Actual 2026 Novel Pitch vs. Suggestions — {name}<br><sup>Batter View</sup>', x=0.5, xanchor='center'),
        xaxis_title='Horizontal Break (ft)', yaxis_title='Induced Vertical Break (ft)',
        xaxis=grid, yaxis=dict(**grid, scaleanchor='x', scaleratio=1),
        legend=dict(x=1.18, y=1, xanchor='left'), height=600, margin=dict(r=220))
    return fig


def validate_pitcher(name):
    """App-style profile + Arsenal/Suggestions info, with the actual 2026 novel pitch alongside."""
    actual = novel_2026[novel_2026['player_name'] == name]
    pit_id = int(actual['pitcher'].iloc[0])
    res    = suggest_pitches(pit_id, pools[throws[pit_id]], pitch_type_summ_2125, min_pitches=20,
                             biomech_distance_threshold=1.5, novelty_distance_threshold=1.2, min_comp_usage_pct=0.01)
    is_r   = throws[pit_id] == 'R'

    if res['status'] != 'ok':
        print(f"No suggestions for {name} (status: {res['status']}).")
        display(actual)
        return

    make_validation_fig(res, actual, is_r).show()

    t, s = res['target_pitches'], res['suggestions']
    table = pd.concat([
        pd.DataFrame({'Kind': 'Current', 'Pitch': t['pitch_type'].map(_full_name).values,
                      'Usage': (t['n'] / t['n'].sum()).values, 'MPH': t['release_speed'].values,
                      'HBreak': t['pfx_x'].values, 'IVBreak': t['pfx_z'].values, '# Comps': np.nan}),
        pd.DataFrame({'Kind': 'Suggested', 'Pitch': s['cluster_label'].values, 'Usage': np.nan,
                      'MPH': s['wavg_release_speed'].values, 'HBreak': s['wavg_pfx_x'].values,
                      'IVBreak': s['wavg_pfx_z'].values, '# Comps': s['n_comps'].values.astype(float)}),
        pd.DataFrame({'Kind': 'ACTUAL 2026', 'Pitch': actual['pitch_type'].map(_full_name).values, 'Usage': np.nan,
                      'MPH': actual['release_speed'].values, 'HBreak': actual['pfx_x'].values,
                      'IVBreak': actual['pfx_z'].values, '# Comps': np.nan}),
    ], ignore_index=True)
    display(table.style.format({'Usage': '{:.1%}', 'MPH': '{:.1f}', 'HBreak': '{:.2f}',
                                'IVBreak': '{:.2f}', '# Comps': '{:.0f}'}, na_rep=''))

In [14]:
# Any pitcher in novel_2026 works. Soroka's actual 2026 Cutter landed almost exactly on his
# suggested Cutter centroid; try e.g. 'Helsley, Ryan' or 'May, Dustin' for other cases.
validate_pitcher('Tong, Jonah')

,Kind,Pitch,Usage,MPH,HBreak,IVBreak,# Comps
0,Current,Changeup,27.5%,85.8,-1.13,0.74,
1,Current,Curveball,12.4%,77.5,0.37,-1.37,
2,Current,Four-Seam Fastball,57.4%,95.2,-0.27,1.65,
3,Current,Slider,2.7%,87.1,0.48,0.17,
4,Suggested,Curveball*,,81.4,0.54,-0.61,12
5,Suggested,Sinker*,,94.6,-1.19,1.03,12
6,ACTUAL 2026,Cutter,,92.0,0.16,0.98,
